# Request JADES 2D Image Cutouts

This notebook show how to request the 2D multi-band image cutouts given the RA and DEC of the object.

In [3]:
from astrocut import CutoutFactory, FITSCutout
from astropy.coordinates import SkyCoord
import astropy.units as u

coord  = SkyCoord(ra=53.1234, dec=-27.5678, unit="deg")
bands  = ["F090W", "F115W", "F150W", "F200W", "F277W", "F356W", "F410M", "F444W"]
field  = "goods-s-deep"
dr     = "dr2"

mosaics = [
    f"mast:HLSP/jades/dr3/hlsp_jades_jwst_nircam_{field}_{b.lower()}_{dr}_i2d.fits"
    for b in bands
]

#factory = CutoutFactory()
#cutout_file = factory.cutout_from_file(
cutout_file = FITSCutout(
    input_files = mosaics,       # one cutout FITS with one extension per band
    coordinates = coord,
    cutout_size = 3 * u.arcsec,
    #output_dir  = "./jades_cutouts_cache",
    verbose     = True
)

DEBUG: Coordinates: <SkyCoord (ICRS): (ra, dec) in deg
    (53.1234, -27.5678)> [cutout]
DEBUG: Cutout size: [3. 3.] arcsec [cutout]


FileNotFoundError: [Errno 2] No such file or directory: 'mast:HLSP/jades/dr3/hlsp_jades_jwst_nircam_goods-s-deep_f090w_dr2_i2d.fits'

In [4]:
from astroquery.mast import Observations
from astroquery.mast import MastCutouts  # or Tesscut for HST/JWST mosaics
from astropy.coordinates import SkyCoord
from astropy.io import fits
import astropy.units as u

coord = SkyCoord(ra=53.1234, dec=-27.5678, unit="deg")

# Query MAST for JWST JADES HLSPs at this position
cutouts = MastCutouts.get_cutouts(
    coordinates=coord,
    size=3 * u.arcsec,           # cutout size
    product="HLSP"               # JADES mosaics are HLSPs
)


ImportError: cannot import name 'MastCutouts' from 'astroquery.mast' (/groups/timeifler/jiachuanxu/python_envs/envs/jwst/lib/python3.10/site-packages/astroquery/mast/__init__.py)

In [5]:
from astrocut import MastCutouts

ImportError: cannot import name 'MastCutouts' from 'astrocut' (/groups/timeifler/jiachuanxu/python_envs/envs/jwst/lib/python3.10/site-packages/astrocut/__init__.py)

In [8]:
import numpy as np
import s3fs
from astropy.io import fits
from astropy.wcs import WCS
from astropy.nddata import Cutout2D
from astropy.coordinates import SkyCoord
import astropy.units as u


# JADES DR2 HLSP naming convention on MAST S3
_JADES_S3_TEMPLATE = (
    "s3://stpubdata/hlsp/jades/jwst/nircam/"
    "{field}/{filter_lower}/v2.0/"
    "hlsp_jades_jwst_nircam_{field}_{filter_lower}_v2.0_drz.fits"
)

# Known JADES fields and their approximate RA/DEC centers
_JADES_FIELDS = {
    "goods-s-deep":   (53.1625, -27.7820),
    "goods-s-medium": (53.1625, -27.7820),
    "goods-n-deep":   (189.228, 62.238),
}


def _guess_field(ra, dec):
    """Return the closest JADES field name for the given coordinates."""
    best, best_dist = None, np.inf
    for field, (fra, fdec) in _JADES_FIELDS.items():
        dist = np.hypot(ra - fra, dec - fdec)
        if dist < best_dist:
            best, best_dist = field, dist
    return best


def get_jades_cutout(ra, dec, band, size_arcsec, field=None):
    """
    Request a 2D cutout from a JWST JADES DR2 mosaic hosted on MAST S3.

    Parameters
    ----------
    ra : float
        Right ascension in degrees (J2000).
    dec : float
        Declination in degrees (J2000).
    band : str
        NIRCam filter, e.g. 'F200W' or 'f200w'.
    size_arcsec : float
        Side length of the cutout in arcseconds.
    field : str, optional
        JADES field name (e.g. 'goods-s-deep'). Auto-detected if None.

    Returns
    -------
    cutout_data : 2D np.ndarray or float
        Cutout image array, or np.nan if no data found.
    cutout_wcs : astropy.wcs.WCS or None
        WCS of the cutout, or None if no data found.
    """
    if field is None:
        field = _guess_field(ra, dec)

    s3_path = _JADES_S3_TEMPLATE.format(
        field=field,
        filter_lower=band.lower(),
    )

    # Open the remote FITS file without downloading it fully
    try:
        fs = s3fs.S3FileSystem(anon=True)
        with fs.open(s3_path, "rb") as f:
            with fits.open(f, memmap=False) as hdul:
                # SCI extension for HLSP drizzle products
                ext = "SCI" if "SCI" in [h.name for h in hdul] else 1
                header = hdul[ext].header
                data = hdul[ext].data

    except FileNotFoundError:
        print(f"No JADES mosaic found: {s3_path}")
        return np.nan, None
    except Exception as e:
        print(f"Failed to open mosaic: {e}")
        return np.nan, None

    if data is None:
        return np.nan, None

    wcs = WCS(header, naxis=2)
    coord = SkyCoord(ra=ra, dec=dec, unit="deg", frame="icrs")
    size = u.Quantity(size_arcsec, u.arcsec)

    try:
        cutout = Cutout2D(data, coord, size, wcs=wcs, mode="partial", fill_value=np.nan)
    except Exception as e:
        print(f"Cutout extraction failed: {e}")
        return np.nan, None

    if cutout.data is None or cutout.data.size == 0:
        return np.nan, None

    return cutout.data, cutout.wcs


# --- Example usage ---
if __name__ == "__main__":
    image, wcs = get_jades_cutout(
        ra=53.13728,
        dec=-27.84784235545948,
        band="F200W",
        size_arcsec=5.0,
    )
    if not isinstance(image, float):
        print(f"Cutout shape: {image.shape}")
    else:
        print("No cutout retrieved.")


No JADES mosaic found: s3://stpubdata/hlsp/jades/jwst/nircam/goods-s-deep/f200w/v2.0/hlsp_jades_jwst_nircam_goods-s-deep_f200w_v2.0_drz.fits
No cutout retrieved.
